### Running this notebook

**The visualizations in this notebook will run in [jupyter lab](https://github.com/jupyterlab/jupyterlab#installation), not jupyter notebook. Google colab is not supported either. VS Code notebooks _might_ work but that has not been tested.** See the fastplotlib supported frameworks for more info: https://github.com/fastplotlib/fastplotlib/#supported-frameworks 

In [1]:
import os
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import mesmerize_core as mc
import mesmerize_viz
import fastplotlib as fpl

In [2]:
from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

In [3]:
if os.name == "nt":
    # disable the cache on windows, this will be automatic in a future version
    cnmf_cache.set_maxsize(0)

In [4]:
# This is just a pandas table display formatting option
pd.options.display.max_colwidth = 120

In [5]:
# mc.set_parent_raw_data_path("/home/username/caiman_data/")
# mc.set_parent_raw_data_path('D:/Data/2P')
mc.set_parent_raw_data_path('H:/Analysis/2P/C57_O1M2/10022023')

WindowsPath('H:/Analysis/2P/C57_O1M2/10022023')

In [6]:
from pathlib import Path
# Set the path to the data you want to load
# os.path.normpath
data_path = Path('D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001')
# data_path = 'D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1300-001'
print(f"Load data from {data_path}.") 

# Set the path for the data export
# export_path = data_path
export_path = mc.get_parent_raw_data_path().joinpath('run1/mesmerize')
print(f"Export data to {export_path}.") 

Load data from D:\Data\2P\C57_O1M2\10022023\TSeries-10022023-1300-001.
Export data to H:\Analysis\2P\C57_O1M2\10022023\run1\mesmerize.


In [7]:
# Create export_path if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)

### Concatenate the ome.tif files into a single multi-page tiff file

In [ ]:
# Using the concat_tif.py script to concatenate the tif files. If needed, install libtiff with: pip install pylibtiff
import concat_tif as ct
ct.concatenate_files([data_path], export_path)

### Batch path, this is where caiman outputs will be organized

This can be anywhere, it does not need to be under the parent raw data path.

In [8]:
batch_path = mc.get_parent_raw_data_path().joinpath("mesmerize-batch/batch.pickle")
batch_path

WindowsPath('H:/Analysis/2P/C57_O1M2/10022023/mesmerize-batch/batch.pickle')

### Create a new batch

This creates a new pandas `DataFrame` with the columns that are necessary for mesmerize. In mesmerize we call this the **batch DataFrame**. You can add additional columns relevant to your experiment, but do not modify columns used by mesmerize.

Note that when you create a DataFrame you will need to use `load_batch()` to load it later. You cannot use `create_batch()` to overwrite an existing batch DataFrame

In [9]:
# create a new batch
# df = mc.create_batch(batch_path)
# to load existing batches use `load_batch()`
df = mc.load_batch(batch_path)
df

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (80, 80), 'overlaps': (40, 40), 'max_shifts': (20, 20), 'max_deviation_rigid': 12, 'border_nan'...",{'mean-projection-path': cd2f1fc6-0623-42f9-9bfe-2c0451546c89\cd2f1fc6-0623-42f9-9bfe-2c0451546c89_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:30:49,173.75 sec,None,cd2f1fc6-0623-42f9-9bfe-2c0451546c89
1,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (80, 80), 'overlaps': (40, 40), 'max_shifts': (12, 12), 'max_deviation_rigid': 6, 'border_nan':...",{'mean-projection-path': 18fbb15b-4e77-4504-910e-2538f387c1ae\18fbb15b-4e77-4504-910e-2538f387c1ae_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:33:48,178.44 sec,None,18fbb15b-4e77-4504-910e-2538f387c1ae
2,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (48, 48), 'overlaps': (24, 24), 'max_shifts': (12, 12), 'max_deviation_rigid': 6, 'border_nan':...",{'mean-projection-path': d2c72ba1-8ebc-4c09-9e74-68aa810886be\d2c72ba1-8ebc-4c09-9e74-68aa810886be_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:36:52,183.9 sec,None,d2c72ba1-8ebc-4c09-9e74-68aa810886be
3,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (48, 48), 'overlaps': (24, 24), 'max_shifts': (6, 6), 'max_deviation_rigid': 3, 'border_nan': '...",{'mean-projection-path': 62e28c18-10b4-4fee-b0e4-d86d1794e6a5\62e28c18-10b4-4fee-b0e4-d86d1794e6a5_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:39:59,186.67 sec,None,62e28c18-10b4-4fee-b0e4-d86d1794e6a5


In [10]:
movie_path = mc.get_parent_raw_data_path().joinpath('run1/mesmerize', 'cat_tiff_bt.tiff')
movie_path

WindowsPath('H:/Analysis/2P/C57_O1M2/10022023/run1/mesmerize/cat_tiff_bt.tiff')

### Parameter Gridsearch

Use a `for` loop to add multiple different parameter variants more efficiently.

In [ ]:
# copy the mcorr_params2 dict to make some changes
# new_params = deepcopy(mcorr_params2)
default_params = \
{
  'main':
    {
        'strides': [80, 80],
        'overlaps': [40, 40],
        'max_shifts': [20, 20],
        'max_deviation_rigid': 12,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

new_params_1 = \
{
  'main':
    {
        'strides': [80, 80],
        'overlaps': [40, 40],
        'max_shifts': [12, 12],
        'max_deviation_rigid': 6,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

new_params_2 = \
{
  'main':
    {
        'strides': [48, 48],
        'overlaps': [24, 24],
        'max_shifts': [12, 12],
        'max_deviation_rigid': 6,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

new_params_3 = \
{
  'main':
    {
        'strides': [48, 48],
        'overlaps': [24, 24],
        'max_shifts': [6, 6],
        'max_deviation_rigid': 3,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}


# # some variants of max_shifts
# for shifts in [4, 8, 16]: 
#     for strides in [32, 64, 128]:
#         for max_deviation_rigid in [4, 8, 16]:
#             overlaps = int(strides / 2)
#             # deep copy is the safest way to copy dicts
#             new_params = deepcopy(default_params)

#             # assign the "max_shifts"
#             new_params["main"]["max_shifts"] = (shifts, shifts)
#             new_params["main"]["strides"] = (strides, strides)
#             new_params["main"]["overlaps"] = (overlaps, overlaps)
#             new_params["main"]["max_deviation_rigid"] = max_deviation_rigid

#             df.caiman.add_item(
#               algo='mcorr',
#               item_name=movie_path.stem,
#               input_movie_path=movie_path,
#               params=new_params
#             )

In [ ]:
df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=default_params
)
df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=new_params_1
)
df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=new_params_2
)
df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=new_params_3
)

In [ ]:
df

### Distinguishing parameter variants

We can see that there are many parameter variants, but it is not easy to see the differences in parameters between the rows that have the same `item_name`.

We can use the `caiman.get_params_diffs()` to see the unique parameters between rows with the same `item_name`

In [11]:
diffs = df.caiman.get_params_diffs(algo="mcorr", item_name=df.iloc[0]["item_name"])
diffs

,max_deviation_rigid,max_shifts,overlaps,strides
0,12,"(20, 20)","(40, 40)","(80, 80)"
1,6,"(12, 12)","(40, 40)","(80, 80)"
2,6,"(12, 12)","(24, 24)","(48, 48)"
3,3,"(6, 6)","(24, 24)","(48, 48)"


### Run multiple batch items.

`df.iterrows()` iterates through rows and returns the numerical index and row for each iteration

We can now see that there is one item in the DataFrame. What we called a "item" in `mesmerize-core` DataFrames is technically called a pandas `Series` or row.

In [12]:
for i, row in df.iterrows():
    # If already processed, skip
    if row["outputs"] is not None:
        print(f"Skipping batch item {i}, id {row.uuid}, algo {row.algo}. Already run.") 
        continue
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

Skipping batch item 0, id cd2f1fc6-0623-42f9-9bfe-2c0451546c89, algo mcorr. Already run.
Skipping batch item 1, id 18fbb15b-4e77-4504-910e-2538f387c1ae, algo mcorr. Already run.
Skipping batch item 2, id d2c72ba1-8ebc-4c09-9e74-68aa810886be, algo mcorr. Already run.
Skipping batch item 3, id 62e28c18-10b4-4fee-b0e4-d86d1794e6a5, algo mcorr. Already run.


### Outputs

Load the output information into the DataFrame

In [13]:
df = df.caiman.reload_from_disk()
df

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (80, 80), 'overlaps': (40, 40), 'max_shifts': (20, 20), 'max_deviation_rigid': 12, 'border_nan'...",{'mean-projection-path': cd2f1fc6-0623-42f9-9bfe-2c0451546c89\cd2f1fc6-0623-42f9-9bfe-2c0451546c89_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:30:49,173.75 sec,None,cd2f1fc6-0623-42f9-9bfe-2c0451546c89
1,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (80, 80), 'overlaps': (40, 40), 'max_shifts': (12, 12), 'max_deviation_rigid': 6, 'border_nan':...",{'mean-projection-path': 18fbb15b-4e77-4504-910e-2538f387c1ae\18fbb15b-4e77-4504-910e-2538f387c1ae_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:33:48,178.44 sec,None,18fbb15b-4e77-4504-910e-2538f387c1ae
2,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (48, 48), 'overlaps': (24, 24), 'max_shifts': (12, 12), 'max_deviation_rigid': 6, 'border_nan':...",{'mean-projection-path': d2c72ba1-8ebc-4c09-9e74-68aa810886be\d2c72ba1-8ebc-4c09-9e74-68aa810886be_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:36:52,183.9 sec,None,d2c72ba1-8ebc-4c09-9e74-68aa810886be
3,mcorr,cat_tiff_bt,run1\mesmerize\cat_tiff_bt.tiff,"{'main': {'strides': (48, 48), 'overlaps': (24, 24), 'max_shifts': (6, 6), 'max_deviation_rigid': 3, 'border_nan': '...",{'mean-projection-path': 62e28c18-10b4-4fee-b0e4-d86d1794e6a5\62e28c18-10b4-4fee-b0e4-d86d1794e6a5_mean_projection.n...,2023-12-04T16:27:37,2023-12-04T16:39:59,186.67 sec,None,62e28c18-10b4-4fee-b0e4-d86d1794e6a5


### Build your own visualizations using `fastplotlib`


In [17]:
# first item is just the raw movie
movies = [df.iloc[0].caiman.get_input_movie()]
# movies = []

# subplot titles
subplot_names = ["raw"]
# subplot_names = []

# we will use the mean images later
means = [df.iloc[0].caiman.get_projection("mean")]

# get the param diffs to set plot titles
param_diffs = df.caiman.get_params_diffs("mcorr", item_name=df.iloc[0]["item_name"])

# add all the mcorr outputs to the list
for i, row in df.iterrows():
    # Select which row to display
    # if i == 2 or i == 3:
    # add to the list of movies to plot
    movies.append(row.mcorr.get_output())

    max_shifts = param_diffs.iloc[i]["max_shifts"][0]
    strides = param_diffs.iloc[i]["strides"][0]
    overlaps = param_diffs.iloc[i]["overlaps"][0]

    # subplot title to show dataframe index
    subplot_names.append(f"ix {i}: max_sh: {max_shifts}, str: {strides}, ove: {overlaps}")

    # mean images which we'll use later
    means.append(row.caiman.get_projection("mean"))

# create the widget
mcorr_iw_multiple = fpl.ImageWidget(
    data=movies,  # list of movies
    window_funcs={"t": (np.mean, 1)}, # window functions as a kwarg, this is what the slider was used for in the ready-made viz
    grid_plot_kwargs={"size": (900, 700)},
    names=subplot_names,  # subplot names used for titles
    cmap="gray" #"gnuplot2"
)

mcorr_iw_multiple.show()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\widgets\image.py:74: UserWarning: Invalid 'window size' value for function: <function mean at 0x00000260029AF6D0>, setting 'window size' = None for this function. Valid values are integers >= 3.
  warn(


RFBOutputContext()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\graphics\_features\_base.py:34: UserWarning: converting float64 array to float32
  warn(f"converting {array.dtype} array to float32")


JupyterOutputContext(children=(JupyterWgpuCanvas(css_height='700px', css_width='900px'), IpywidgetToolBar(chil…

Optionally hide the histogram LUT tool

In [18]:
for subplot in mcorr_iw_multiple.gridplot:
    subplot.docks["right"].size = 0

Modify the `window_funcs` at any time. This is what the slider in `mesmerize-viz` does.

In [19]:
mcorr_iw_multiple.window_funcs["t"].window_size = 43

In [20]:
mcorr_iw_multiple.close()

**Mesmerize-viz**

In [ ]:
# mcorr_viz = df.mcorr.viz(
#     data_options=["input", "mcorr"], 
#     start_index=0,
#     image_widget_kwargs={"grid_plot_kwargs": {"size": (1000, 500)}}  # you can also pass kwargs to the ImageWidget that is created
# )
# mcorr_viz.show(sidecar=True) #
# mcorr_viz.image_widget.cmap = "gray"

In [ ]:
# # close when done
# mcorr_viz.close()

### Optional, cleanup DataFrame

ix `6` seems to work the best so we will cleanup the DataFrame and remove all other items.

Remove batch items (i.e. rows) using `df.caiman.remove_item(<item_uuid>)`. This also cleans up the output data in the batch directory.

In [21]:
# make a list of rows we want to keep using the uuids
rows_keep = [df.iloc[2].uuid]
rows_keep

['d2c72ba1-8ebc-4c09-9e74-68aa810886be']

**Note:** On windows calling `remove_item()` will raise a `PermissionError` if you have the memmap file open. The workaround is to shutdown the current kernel and then use `df.caiman.remove_item()`. For example, you can keep another notebook that you use just for cleaning unwanted mcorr items.

There is currently no way to close a `numpy.memmap`: https://github.com/numpy/numpy/issues/13510

In [22]:
for i, row in df.iterrows():
    if row.uuid not in rows_keep:
         df.caiman.remove_item(row.uuid)

df

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\mesmerize_core\caiman_extensions\common.py:220: FutureWarning: You are trying to use the following experimental feature, this may change in the future without warning:
CaimanDataFrameExtensions.get_children
This feature will change in the future and directly return the  a DataFrame of children (rows, ie. child batch items row) instead of a list of UUIDs

  children = self.get_children(index)


PermissionError: You do not have permissions to remove the output data for the batch item, aborting.

Indices are always reset when you use `caiman.remove_item()`. UUIDs are always preserved.

### CNMF

Perform CNMF using the mcorr output.

Similar to mcorr, put the CNMF params within the `main` key. The `refit` key will perform a second iteration, as shown in the `caiman` `demo_pipeline.ipynb` notebook.

In [ ]:
# some params for CNMF
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 30, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.85,
            'rf': 15,
            'stride': 6, # "stride" for cnmf, "strides" for mcorr
            'K': 4,
            'gSig': [4, 4],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.0,
            'rval_thr': 0.7,
            'use_cnn': True,
            'min_cnn_thr': 0.8,
            'cnn_lowest': 0.1,
            'decay_time': 0.4,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

### Add CNMF item

You can provide the mcorr item row to `input_movie_path` and it will resolve the path of the input movie from the entry in the DataFrame.

In [ ]:
good_mcorr_index = 6

# add a batch item
df.caiman.add_item(
    algo='cnmf', # algo is cnmf
    input_movie_path=df.iloc[good_mcorr_index],  # use mcorr output from a completed batch item
    params=params_cnmf,
    item_name=df.iloc[good_mcorr_index]["item_name"], # use the same item name
)

See the cnmf item at the bottom of the dataframe

In [ ]:
df

### Run CNMF

The API is identical to running mcorr

In [ ]:
index = -1  # most recently added item
df.iloc[index].caiman.run()

### Reload dataframe

In [ ]:
df = df.caiman.reload_from_disk()
df

### CNMF outputs

Similar to mcorr, you can use the `mesmerize-core` API to fetch the outputs. The API reference for CNMF is here: https://mesmerize-core.readthedocs.io/en/latest/api/cnmf.html

In [ ]:
index = -1  # index of the cnmf item, last item in the dataframe

# temporal components
temporal = df.iloc[index].cnmf.get_temporal()

In [ ]:
temporal.shape

Many of the cnmf functions take a rich set of arguments

In [ ]:
# get accepted or rejected components
temporal_good = df.iloc[index].cnmf.get_temporal("good")

# shape is [n_components, n_frames]
temporal_good.shape

In [ ]:
# get specific components
df.iloc[index].cnmf.get_temporal(np.array([1, 5, 9]))

In [ ]:
# get temporal with the residuals, i.e. C + YrA
temporal_with_residuals = df.iloc[index].cnmf.get_temporal(add_residuals=True)

In [ ]:
# get contours
contours = df.iloc[index].cnmf.get_contours()

Returns: `(list of np.ndarray of contour coordinates, list of center of mass)`

In [ ]:
print(f"contour 0 coordinates:\n\n{contours[0][0]}\n\n com: {contours[1][0]}")

In [ ]:
len(contours)

In [ ]:
# get_contours() also takes arguments
contours_good = df.iloc[index].cnmf.get_contours("good")

In [ ]:
len(contours_good[0]) # number of contours

swap_dim

In [ ]:
# get the first contour using swap_dim=True (default)
swap_dim_true = df.iloc[index].cnmf.get_contours()[0][0]

In [ ]:
# get the first contour  with swap_dim=False
swap_dim_false = df.iloc[index].cnmf.get_contours(swap_dim=False)[0][0]

In [ ]:
plt.plot(
    swap_dim_true[:, 0], 
    swap_dim_true[:, 1], 
    label="swap_dim=True"
)
plt.plot(
    swap_dim_false[:, 0], 
    swap_dim_false[:, 1], 
    label="swap_dim=False"
)
plt.legend()

In [ ]:
# swap_dim swaps the x and y dims
plt.plot(
    swap_dim_true[:, 0], 
    swap_dim_true[:, 1], 
    linewidth=30
)
plt.plot(
    swap_dim_false[:, 1], 
    swap_dim_false[:, 0], 
    linewidth=10
)

### Reconstructed movie - `A * C`
### Reconstructed background - `b * f`
### Residuals - `Y - AC - bf` 

Mesmerize-core provides these outputs as lazy arrays. This allows you to work with arrays that would otherwise be hundreds of gigabytes or terabytes in size.

In [ ]:
rcm = df.iloc[index].cnmf.get_rcm()
rcm

LazyArrays behave like numpy arrays

In [ ]:
rcm[42]

In [ ]:
rcm[10:20].shape

### Using LazyArrays

In [ ]:
rcm_accepted = df.iloc[index].cnmf.get_rcm("good")
rcm_rejected = df.iloc[index].cnmf.get_rcm("bad")

In [ ]:
iw_max = fpl.ImageWidget(
    data=[rcm_accepted.max_image, rcm_rejected.max_image],
    names=["accepted", "rejected"],
    grid_plot_kwargs={"size": (900, 450)},
    cmap="gnuplot2"
)
iw_max.show()

In [ ]:
iw_rcm_separated = fpl.ImageWidget(
    data=[rcm_accepted, rcm_rejected],
    names=["accepted", "rejected"],
    grid_plot_kwargs={"size": (900, 450)},
    cmap="gnuplot2"
)
iw_rcm_separated.show()

### All CNMF LazyArrays

In [ ]:
rcb = df.iloc[index].cnmf.get_rcb()
residuals = df.iloc[index].cnmf.get_residuals()
input_movie = df.iloc[index].caiman.get_input_movie()

`ImageWidget` accepts arrays that are sufficiently numpy-like

In [ ]:
iw_rcm = fpl.ImageWidget(
    data=[input_movie, rcm, rcb, residuals], 
    grid_plot_kwargs={"size": (800, 600)}, 
    cmap="gnuplot2"
)
iw_rcm.show()

In [ ]:
iw_rcm.close()

### Visualize everything with `mesmerize-viz`

In [ ]:
viz_cnmf = df.cnmf.viz()
viz_cnmf.show()

In [ ]:
viz_cnmf.close()

### Parameter Gridsearch

As shown for motion correction, the purpose of `mesmerize-core` is to perform parameter searches

In [ ]:
# itertools.product makes it easy to loop through parameter variants
# basically, it's easier to read that deeply nested for loops
from itertools import product

# variants of several parameters
gSig_variants = [4, 6]
K_variants = [4, 8]
merge_thr_variants = [0.8, 0.95]

# always use deepcopy like before
new_params_cnmf = deepcopy(params_cnmf)

# create a parameter grid
parameter_grid = product(gSig_variants, K_variants, merge_thr_variants)

# a single for loop to go through all the various parameter combinations
for gSig, K, merge_thr in parameter_grid:
    # deep copy params dict just like before
    new_params_cnmf = deepcopy(new_params_cnmf)
    
    new_params_cnmf["main"]["gSig"] = [gSig, gSig]
    new_params_cnmf["main"]["K"] = K
    new_params_cnmf["main"]["merge_thr"] = merge_thr
    
    # add param combination variant to batch
    df.caiman.add_item(
        algo="cnmf",
        item_name=df.iloc[good_mcorr_index]["item_name"],  # good mcorr item
        input_movie_path=df.iloc[good_mcorr_index],
        params=new_params_cnmf
    )

We now have lot of cnmf items

In [ ]:
df

View the diffs

In [ ]:
df.caiman.get_params_diffs(algo="cnmf", item_name=df.iloc[-1]["item_name"])

### Run the `cnmf` batch items

In [ ]:
for i, row in df.iterrows():
    if row["outputs"] is not None: # item has already been run
        continue # skip
        
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

### Load outputs

In [ ]:
df = df.caiman.reload_from_disk()

In [ ]:
df

### Visualize with `mesmerize-viz`

In [ ]:
viz_cnmf = df.cnmf.viz()
viz_cnmf.show(sidecar=True)

### Caiman docs on component eval

https://caiman.readthedocs.io/en/latest/Getting_Started.html#component-evaluation

> The quality of detected components is evaluated with three parameters:
>
> Spatial footprint consistency (rval): The spatial footprint of the component is compared with the frames where this component is active. Other component’s signals are subtracted from these frames, and the resulting raw data is correlated against the spatial component. This ensures that the raw data at the spatial footprint aligns with the extracted trace.
>
> Trace signal-noise-ratio (SNR): Peak SNR is calculated from strong calcium transients and the noise estimate.
>
> CNN-based classifier (cnn): The shape of components is evaluated by a 4-layered convolutional neural network trained on a manually annotated dataset. The CNN assigns a value of 0-1 to each component depending on its resemblance to a neuronal soma.
> 
> Each parameter has a low threshold:
> - (rval_lowest (default -1), SNR_lowest (default 0.5), cnn_lowest (default 0.1))
>
> and high threshold
> 
> - (rval_thr (default 0.8), min_SNR (default 2.5), min_cnn_thr (default 0.9))
> 
> A component has to exceed ALL low thresholds as well as ONE high threshold to be accepted.

### This rich visualization is still customizable!

Public attributes:

- `image_widget`: the `ImageWidget` in the visualization
- `plot_temporal`: the `Plot` with the temporal
- `plot_heatmap`: the `Plot` with the heatmap
- `cnmf_obj`: The cnmf object currently being visualized. This object gets saved to disk when you click the "Save Eval to disk" button.
- `component_index`: current component index, `int`

A few public methods:
- `show()` show the visualization
- `set_component_index(index: int)` manually set the component index

In [ ]:
viz_cnmf.image_widget.cmap = "gray"

In [ ]:
viz_cnmf.plot_heatmap

In [ ]:
viz_cnmf.plot_heatmap["heatmap"].cmap = "viridis"

In [ ]:
viz_cnmf.plot_heatmap["heatmap"].cmap.vmax

In [ ]:
viz_cnmf.plot_heatmap["heatmap"].cmap.vmax = 2_000

Customize contours

In [ ]:
for subplot in viz_cnmf.image_widget.gridplot:
    subplot["contours"][:].thickness = 1.0

In [ ]:
for subplot in viz_cnmf.image_widget.gridplot:
    subplot["contours"].visible = False

In [ ]:
for subplot in viz_cnmf.image_widget.gridplot:
    subplot["contours"].visible = True

In [ ]:
viz_cnmf.plot_temporal["line"].thickness()

In [ ]:
viz_cnmf.plot_temporal["line"].thickness = 1

### Visualize fewer things

In [ ]:
viz_simple = df.cnmf.viz(
    image_data_options=["input", "rcm"],
)
viz_simple.show(sidecar=True)

### More customization of kwargs

In [ ]:
viz_more_custom = df.cnmf.viz(
    image_data_options=["input", "rcm", "rcm-max", "corr"],
    temporal_kwargs={"add_residuals": True},
)

In [ ]:
viz_more_custom.show(sidecar=True)